In [113]:
import spacy
import networkx as nx
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import holidays
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

c:\NCKH NLP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [82]:
file_path = 'Data.csv'
data = pd.read_csv(file_path)


In [83]:
data.head()

,href,title,publish_date,content,keyword
0,https://cafef.vn/lai-suat-ngan-hang-lpbank-moi...,Lãi suất ngân hàng LPBank mới nhất tháng 11/20...,05-11-2024 - 20:04 PM,TIN MỚIẢnh minh họaLãi suất tiết kiệm tại quầy...,Lãi suất ngân hàng
1,https://cafef.vn/lai-suat-ngan-hang-hdbank-moi...,Lãi suất ngân hàng HDBank mới nhất tháng 11/20...,05-11-2024 - 20:00 PM,TIN MỚIảnh minh họaLãi suất huy động dành cho ...,Lãi suất ngân hàng
2,https://cafef.vn/lai-suat-ngan-hang-shb-moi-nh...,Lãi suất ngân hàng SHB mới nhất tháng 10/2024:...,05-10-2024 - 15:05 PM,TIN MỚIẢnh minh họaLãi suất tiền gửi tại quầy ...,Lãi suất ngân hàng
3,https://cafef.vn/lai-suat-ngan-hang-quan-doi-m...,Lãi suất Ngân hàng Quân đội (MB) tháng 10/2024...,05-10-2024 - 13:02 PM,TIN MỚIẢnh minh họaLãi suất tiết kiệm dành cho...,Lãi suất ngân hàng
4,https://cafef.vn/lai-suat-ngan-hang-hdbank-moi...,Lãi suất ngân hàng HDBank mới nhất tháng 10/20...,04-10-2024 - 00:30 AM,TIN MỚILãi suất huy động dành cho khách hàng g...,Lãi suất ngân hàng


In [84]:
an_rows = data[data.isna().any(axis=1)]
an_rows

,href,title,publish_date,content,keyword
201,https://cafef.vn/emagazine-pho-thong-doc-thuon...,[eMagazine] Phó Thống đốc Thường trực Ngân hàn...,NaN,No Content Found,Chính sách tiền tệ
327,https://cafef.vn/big-story/toan-canh-loi-nhuan...,"Toàn cảnh lợi nhuận ngân hàng quý 2/2021: Ba ""...",NaN,No Content Found,Lợi nhuận ngân hàng
804,https://cafef.vn/san-bay-long-thanh-trien-khai...,Sân bay Long Thành triển khai rầm rộ nhưng giá...,NaN,No Content Found,Giá bất động sản
807,https://cafef.vn/du-an-duong-vanh-dai-75000-ty...,"Dự án đường vành đai 75.000 tỷ chậm tiến độ, g...",NaN,No Content Found,Giá bất động sản
809,https://cafef.vn/cau-vuot-mai-dich-sap-hoan-th...,"Cầu vượt Mai Dịch sắp hoàn thành, giá bất động...",NaN,No Content Found,Giá bất động sản
1230,https://cafef.vn/big-story/dhcd-shb-tang-von-t...,"ĐHCĐ SHB: Tăng vốn điều lệ thêm hơn 5.500 tỷ, ...",NaN,No Content Found,mở rộng thị trường


In [85]:
data.dtypes

href            object
title           object
publish_date    object
content         object
keyword         object
dtype: object

In [86]:
unique_publish_dates = data['publish_date'].unique()
unique_publish_dates

array(['05-11-2024 - 20:04 PM', '05-11-2024 - 20:00 PM',
       '05-10-2024 - 15:05 PM', ..., '05-08-2008 - 07:51 AM',
       '28-07-2008 - 08:31 AM', '26-07-2008 - 09:45 AM'], dtype=object)

In [87]:
def convert_publish_date(data, column_name='publish_date'):
    data[column_name] = data[column_name].astype(str)

    data[column_name] = data[column_name].str.replace('AM', '', regex=False)
    data[column_name] = data[column_name].str.replace('PM', '', regex=False)

    data[column_name] = data[column_name].str.strip()

    data[column_name] = pd.to_datetime(data[column_name], format='%d-%m-%Y - %H:%M', errors='coerce')
    
    return data

In [88]:
data = convert_publish_date(data, column_name='publish_date')

In [89]:
def clean_data(data):
    
    data = data.dropna()
    
    data['content'] = data['content'].str.replace('TIN MỚI', '', regex=False)
    data['content'] = data['content'].str.replace('Ảnh minh họa', '', regex=False)
    
    data = data.drop_duplicates(subset=['href'])
    
    data['title'] = data['title'].str.strip()
    data['content'] = data['content'].str.strip()
    data['keyword'] = data['keyword'].str.strip()
    
    return data


In [108]:
cleaned_data = clean_data(data)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_78640\1989075551.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['content'] = data['content'].str.replace('TIN MỚI', '', regex=False)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_78640\1989075551.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['content'] = data['content'].str.replace('Ảnh minh họa', '', regex=False)


In [109]:
cleaned_data.dtypes

href                    object
title                   object
publish_date    datetime64[ns]
content                 object
keyword                 object
dtype: object

In [110]:
cleaned_data['day'] = cleaned_data['publish_date'].dt.date
cleaned_data['time'] = cleaned_data['publish_date'].dt.strftime('%H:%M:%S')

In [111]:
cleaned_data.dtypes

href                    object
title                   object
publish_date    datetime64[ns]
content                 object
keyword                 object
day                     object
time                    object
dtype: object

In [104]:
cleaned_data.to_csv('new_data.csv', index=False)

In [112]:
#Hour of the Day
cleaned_data['Hour_of_the_day'] = pd.to_datetime(cleaned_data['time']).dt.hour

#Is Holiday
vn_holidays = holidays.Vietnam()
cleaned_data['Is_holiday'] = cleaned_data['day'].apply(lambda x: 1 if x in vn_holidays else 0)

# Number of News Articles (per day)
cleaned_data['Number of news articles'] = cleaned_data.groupby(cleaned_data['publish_date'].dt.date)['publish_date'].transform('count')


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_78640\1064200690.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned_data['Hour_of_the_day'] = pd.to_datetime(cleaned_data['time']).dt.hour


In [114]:
# Load PhoBERT

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=3)  # num_labels = 3 nếu bạn muốn phân loại positive, negative và neutral

def classify_sentiment_phobert(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    sentiment = torch.argmax(probs).item()  # -1 cho negative, 0 cho neutral, 1 cho positive
    if sentiment == 1:
        return "positive"
    elif sentiment == -1:
        return "negative"
    elif sentiment == 0:
        return "neutral"
    
cleaned_data['sentiment'] = cleaned_data['content'].apply(classify_sentiment_phobert)

c:\NCKH NLP\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--vinai--phobert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly i

In [115]:
daily_sentiment_counts = cleaned_data.groupby([cleaned_data['publish_date'].dt.date, 'sentiment']).size().unstack(fill_value=0)
daily_sentiment_counts.columns = ['Negative articles', 'Positive articles']

In [116]:
data_vn30_path = 'data_vn30_1.csv'
data_vn30 = pd.read_csv(data_vn30_path)

In [117]:
vn30_mwe = data_vn30['Mã Cổ phiếu'].tolist() + data_vn30['Tên Công ty'].tolist()
interest_rate_mwe = ["lãi suất", "tăng lãi suất", "giảm lãi suất", "lãi suất cơ bản"]
inflation_mwe = ["lạm phát", "tăng giá", "giảm phát", "tăng chi phí sinh hoạt"]
global_market_mwe = ["thị trường toàn cầu", "suy thoái kinh tế", "kinh tế thế giới", "thị trường chứng khoán quốc tế"]

mwe_dict = {
    'VN30_sentiment': vn30_mwe,
    'interest_rate_sentiment': interest_rate_mwe,
    'inflation_sentiment': inflation_mwe,
    'global_market_sentiment': global_market_mwe
}

def calculate_sentiment_phobert(text, mwes):
    sentiment_score = 0
    count = 0
    for mwe in mwes:
        if mwe in text:
            inputs = tokenizer(mwe, return_tensors="pt", truncation=True, padding=True)
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            sentiment = torch.argmax(probs).item()  # 0: negative, 1: neutral, 2: positive
            
            # Chuyển đổi sentiment về -1, 0, 1
            if sentiment == 2:
                sentiment_score += 1  # Positive
            elif sentiment == 1:
                sentiment_score += 0  # Neutral
            elif sentiment == 0:
                sentiment_score -= 1  # Negative
            count += 1
    if count == 0:
        return 0  # Không có MWE nào khớp
    return sentiment_score / count 


cleaned_data['VN30_sentiment'] = cleaned_data['content'].apply(lambda x: calculate_sentiment_phobert(x, mwe_dict['VN30_sentiment']))
cleaned_data['interest_rate_sentiment'] = cleaned_data['content'].apply(lambda x: calculate_sentiment_phobert(x, mwe_dict['interest_rate_sentiment']))
cleaned_data['inflation_sentiment'] = cleaned_data['content'].apply(lambda x: calculate_sentiment_phobert(x, mwe_dict['inflation_sentiment']))
cleaned_data['global_market_sentiment'] = cleaned_data['content'].apply(lambda x: calculate_sentiment_phobert(x, mwe_dict['global_market_sentiment']))

cleaned_data['day'] = pd.to_datetime(cleaned_data['publish_date']).dt.date
daily_sentiment = cleaned_data.groupby('day').agg({
    'VN30_sentiment': 'mean',
    'interest_rate_sentiment': 'mean',
    'inflation_sentiment': 'mean',
    'global_market_sentiment': 'mean'
}).reset_index()


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [118]:
cleaned_data.head()

,href,title,publish_date,content,keyword,day,time,Hour_of_the_day,Is_holiday,Number of news articles,sentiment,VN30_sentiment,interest_rate_sentiment,inflation_sentiment,global_market_sentiment
0,https://cafef.vn/lai-suat-ngan-hang-lpbank-moi...,Lãi suất ngân hàng LPBank mới nhất tháng 11/20...,2024-11-05 20:04:00,Lãi suất tiết kiệm tại quầy LPBank dành cho kh...,Lãi suất ngân hàng,2024-11-05,20:04:00,20,0,7,neutral,0.0,0.0,0.0,0.0
1,https://cafef.vn/lai-suat-ngan-hang-hdbank-moi...,Lãi suất ngân hàng HDBank mới nhất tháng 11/20...,2024-11-05 20:00:00,ảnh minh họaLãi suất huy động dành cho khách h...,Lãi suất ngân hàng,2024-11-05,20:00:00,20,0,7,neutral,-1.0,0.0,0.0,0.0
2,https://cafef.vn/lai-suat-ngan-hang-shb-moi-nh...,Lãi suất ngân hàng SHB mới nhất tháng 10/2024:...,2024-10-05 15:05:00,Lãi suất tiền gửi tại quầy SHB tháng 10/2024Vớ...,Lãi suất ngân hàng,2024-10-05,15:05:00,15,0,6,neutral,1.0,0.0,0.0,0.0
3,https://cafef.vn/lai-suat-ngan-hang-quan-doi-m...,Lãi suất Ngân hàng Quân đội (MB) tháng 10/2024...,2024-10-05 13:02:00,Lãi suất tiết kiệm dành cho khách hàng cá nhân...,Lãi suất ngân hàng,2024-10-05,13:02:00,13,0,6,neutral,0.0,0.0,0.0,0.0
4,https://cafef.vn/lai-suat-ngan-hang-hdbank-moi...,Lãi suất ngân hàng HDBank mới nhất tháng 10/20...,2024-10-04 00:30:00,Lãi suất huy động dành cho khách hàng gửi tiền...,Lãi suất ngân hàng,2024-10-04,00:30:00,0,0,2,neutral,-1.0,0.0,0.0,0.0


In [119]:
daily_sentiment.head()

,day,VN30_sentiment,interest_rate_sentiment,inflation_sentiment,global_market_sentiment
0,2007-11-10,0.0,0.0,0.0,0.0
1,2007-11-22,0.0,0.0,0.0,0.0
2,2008-03-12,0.0,0.0,0.0,0.0
3,2008-07-04,-1.0,0.0,0.0,0.0
4,2008-07-09,0.0,0.0,0.0,0.0


In [120]:
cleaned_data.to_csv('final_data_1.csv', index=False)